In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Hemolytik)

This notebook curates the **Hemolytik** dataset by integrating hemolytic peptide sequences from multiple textual sources and protein structure files. The source includes natural peptides, non-natural (chemically modified) variants, and peptide sequences derived from PDB structures. Here we standardize all inputs, separate natural and modified sequences, perform duplicate consistency checks, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** Hemolytik
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple sequence-based sources** provided by Hemolytik:
  - natural peptide datasets (`naturalseqD`, `naturalseqL`, `naturalseqmix`),
  - non-natural peptide datasets (`nonnaturalseqD`, `nonnaturalseqL`, `nonnaturalseqmix`),
  - a comprehensive annotation table (`allsequences.txt`).
- **Extracts peptide sequences from PDB structures**:
  - parses all `.pdb` files in the `allstructures/all` directory,
  - converts standard residues to one-letter amino acid codes,
  - treats each extracted chain as a separate peptide sequence.
- **Separates sequences into natural and modified subsets**:
  - *non-modified*: free N- and C-termini and no non-natural modifications,
  - *modified*: any N-terminal, C-terminal, or non-natural modification.
- **Builds unified datasets**:
  - all sequences are normalized to uppercase,
  - all entries are assigned a positive hemolytic label (`label = 1`),
  - modified sequences retain an explicit `modification` annotation.
- **Checks duplicated sequences** independently for natural and modified datasets:
  - unique sequences are preserved,
  - duplicates with consistent labels are collapsed,
  - conflicting cases (if any) are reported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv` (natural sequences),
  - `modified_hemolytic_dataset.csv` (modified sequences with annotations),
  - `metadata.json`.

In [2]:
name_source = "Hemolytik"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_naturalseqD = pd.read_csv(f"{PATH_INPUT}/{name_source}/naturalseqD.txt", sep="\t")

In [4]:
df_naturalseqL = pd.read_csv(f"{PATH_INPUT}/{name_source}/naturalseqL.txt", sep="\t")

In [5]:
df_naturalseqmix = pd.read_csv(f"{PATH_INPUT}/{name_source}/naturalseqmix.txt", sep="\t")

In [6]:
df_nonnaturalseqD = pd.read_csv(f"{PATH_INPUT}/{name_source}/nonnaturalseqD.txt", sep="\t")

In [7]:
df_nonnaturalseqL = pd.read_csv(f"{PATH_INPUT}/{name_source}/nonnaturalseqL.txt", sep="\t")

In [8]:
df_nonnaturalseqmix = pd.read_csv(f"{PATH_INPUT}/{name_source}/nonnaturalseqmix.txt", sep="\t")

In [9]:
df_allsequences = pd.read_csv(f"{PATH_INPUT}/{name_source}/allsequences.txt", sep="\t")

In [10]:
pdb_dir = f"{PATH_INPUT}/{name_source}/allstructures/all"
records = []

for pdb_file in Path(pdb_dir).glob("*.pdb"):
    sequences = extract_sequence_from_pdb_notstandar(str(pdb_file))

    for chain_id, seq in sequences.items():
        records.append({
            "ID": pdb_file.stem,
            "SEQ": seq
        })

df_structures = pd.DataFrame(records)

In [11]:
mask_no_mod = (df_allsequences["Nter-modification"] == "Free") & (df_allsequences["Cter-modification"] == "Free") & (df_allsequences["Non-natural-modification"].isna())
df_non_modified_allseq = (
    df_allsequences.loc[mask_no_mod, ["ID","SEQ"]]
    .reset_index(drop=True)
)

In [12]:
df_modified_all_seq = (
    df_allsequences
    .loc[
        ~mask_no_mod,
        ["SEQ", "Nter-modification", "Cter-modification", "Non-natural-modification"]
    ]
    .assign(
        **{
            "Non-natural-modification": lambda d: d[
                ["Nter-modification", "Cter-modification", "Non-natural-modification"]
            ].apply(
                lambda row: ";".join(
                    str(x).strip() for x in row
                    if pd.notna(x) and str(x).strip() != ""
                ),
                axis=1
            )
        }
    )
    [["SEQ", "Non-natural-modification"]]
    .reset_index(drop=True)
)

- Concatenating dataset

In [13]:
df_non_modified = (
    pd.concat(
        [
            df_naturalseqD,
            df_naturalseqL,
            df_naturalseqmix,
            df_non_modified_allseq,
            df_structures,
        ],
        ignore_index=True,
    )
    .rename(columns={"SEQ": "sequence"})
    .assign(
        sequence=lambda d: d["sequence"].str.upper(),
        label=1
    )
    [["sequence", "label"]]
)

df_non_modified.shape

(6526, 2)

In [14]:
df_modified = (
    pd.concat(
        [
            df_nonnaturalseqD,
            df_nonnaturalseqL,
            df_nonnaturalseqmix,
            df_modified_all_seq
        ],
        ignore_index=True,
    )
    .rename(columns={"SEQ": "sequence",
                     "Non-natural-modification": "modification"})
    .assign(
        label=1
    )
    [["sequence", "label", "modification"]]
)

df_modified.shape

(1867, 3)

- Checking duplicates

In [15]:
df_remove_duplicated_nonmodified, df_errors_nonmodified, df_unique_nonmodified = processing_duplicated(df_non_modified, group_seq="sequence", sort_key="label")
df_full_nonmodified = pd.concat([df_unique_nonmodified, df_remove_duplicated_nonmodified], axis=0)

In [16]:
df_errors_nonmodified.shape

(0, 1)

In [17]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(df_modified, group_seq="sequence", sort_key="label")
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)

In [18]:
df_errors_mod.shape

(0, 1)

- Working with metada

In [19]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [20]:
raw_total_sequences = (
    len(df_naturalseqD)
    + len(df_naturalseqL)
    + len(df_naturalseqmix)
    + len(df_nonnaturalseqD)
    + len(df_nonnaturalseqL)
    + len(df_nonnaturalseqmix)
    + len(df_allsequences)
    + len(df_structures)
)

In [21]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full_nonmodified),
    "number_of_positive_sequences": int((df_full_nonmodified["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full_nonmodified["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors_nonmodified),
    "number_of_modified_sequences" : len(df_full_mod),
    "number_of_erroneous_modified_sequences" : len(df_errors_mod),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2014,
 'last update date': datetime.datetime(2014, 1, 1, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'tsv;PDB',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'http://crdd.osdd.net/raghava/hemolytik/dwn_seq.php',
 'publication': 'https://academic.oup.com/nar/article/42/D1/D444/1041683?login=false',
 'number_of_raw_sequences': 8393,
 'number_of_sequences_retained': 2471,
 'number_of_positive_sequences': 2471,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'number_of_modified_sequences': 1097,
 'number_of_erroneous_modified_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [22]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [23]:
df_full_nonmodified.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)